# 🤖 Build Your First AI Agent — Hands-on Codelab
### Agentic AI with Google ADK (Agent Development Kit) → Evaluate → Go live on Cloud Run

**Time:** ~2 hours · **Level:** absolute beginner (if you can run a cell, you can do this!)

By the end you will have built **Campus Buddy** 🎒 — an AI agent that can check your timetable, read the canteen menu, calculate your CGPA and remember things for you.

| Part | What you do | ⏱ |
|---|---|---|
| 0 | Setup: install ADK + get a free Groq API key | 10 min |
| 1 | Chatbot vs Agent — see the difference | 10 min |
| 2 | Your first agent (no tools) | 10 min |
| 3 | Give it tools 🛠️ (this is where it becomes an *agent*) | 20 min |
| 4 | Sessions, State & Events — the agent's memory | 15 min |
| 5 | ADK Dev UI — Events, Trace, State, Sessions | 15 min |
| 6 | Evaluation with a **golden dataset** 🥇 | 20 min |
| 7 | Go live on **Cloud Run** 🚀 | 15 min |
| 8 | Challenges 🏆 | rest |

> 💡 **How to use this notebook:** run cells top to bottom with **Shift + Enter**. When you see 🧠 **Predict first**, guess the answer *before* running. When you see ✏️ **Your turn**, change the code yourself.

---
## Part 0 · Setup (10 min)

**Step 1 — Get a free Groq API key** ⚡ (Groq runs open models like Llama super fast)
1. Open 👉 https://console.groq.com/keys and sign in (Google login works).
2. Click **Create API Key** → copy it (it starts with `gsk_`).
3. In Colab, click the 🔑 **Secrets** icon on the left sidebar → **Add new secret** → Name: `GROQ_API_KEY`, Value: *your key* → turn on **Notebook access**.

> ℹ️ ADK is made by Google but works with many LLMs. We connect it to Groq through **LiteLLM**, a library that speaks to 100+ model providers.

> 🔐 Never paste your key directly into code you share. Secrets keep it private.

**Step 2 — Install ADK** (takes ~1 minute)

In [28]:
!pip install -q "google-adk[eval]" "litellm>=1.84"
!adk --version

adk, version 2.11.0


In [29]:
import os
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")

from google.adk.models.lite_llm import LiteLlm
MODEL_NAME = "groq/qwen/qwen3.8-27b"   # provider/model — try "groq/openai/gpt-oss-120b" too
# max_tokens keeps each call under Groq's free-tier tokens-per-minute limit; num_retries auto-retries on 429s
MODEL = LiteLlm(model=MODEL_NAME, max_tokens=300, num_retries=5)   # the agent's brain 🧠
print("✅ Key loaded:", os.environ["GROQ_API_KEY"][:6] + "...")

✅ Key loaded: gsk_J7...


---
## Part 1 · Chatbot vs Agent (10 min)

An **LLM (chatbot)** can only *talk*. It answers from what it learned during training.

An **AI Agent** = **LLM (brain) 🧠 + Tools (hands) 🛠️ + Memory 📒 + a Goal 🎯**. It can *decide* to take actions — look things up, calculate, save data — then answer.

🧠 **Predict first:** if we ask a plain LLM *"What classes do I have on Monday?"* — what will it say?

In [30]:
import litellm
reply = litellm.completion(model=MODEL_NAME,
                           messages=[{"role": "user", "content": "What classes do I have on Monday?"}])
print(reply.choices[0].message.content)

I don't have access to your personal schedule, calendar, or university/institution portal, so I cannot tell you specifically what classes you have on Monday.

To find out, you can:

1. **Check your student portal or academic planner** (e.g., LMS like Canvas, Blackboard, or your school’s registration system).
2. **Look at your syllabi** from the beginning of the semester.
3. **Check your digital calendar** (e.g., Google Calendar, Outlook, or Apple Calendar) where you may have added your classes.
4. **Refer to your printed or saved timetable/course schedule** provided by your institution.

If you’d like help organizing or analyzing your schedule once you have it, feel free to share your class times and days (without sensitive personal information), and I can help you plan your week!


👆 It doesn't know! It has no access to **your** college's timetable. It either apologises or (worse) **makes something up** — that's called a *hallucination*.

Agents fix this by giving the LLM **tools** to fetch real data. Let's build one.

---
## Part 2 · Your first ADK agent (10 min)

An ADK agent needs just 4 things:

| Field | Meaning | Analogy |
|---|---|---|
| `name` | unique id | your roll number |
| `model` | the LLM brain | the student's brain |
| `instruction` | how to behave | the class rules |
| `tools` | functions it can call | calculator, notebook, phone |

ADK also needs a **Runner** (the engine that runs the conversation) and a **Session service** (where conversations are stored).

In [31]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

hello_agent = Agent(
    name="hello_agent",
    model=MODEL,
    instruction="You are a cheerful senior student. Welcome juniors to college in 2 short sentences.",
)

# A tiny helper we'll reuse: send a message and PRINT EVERY STEP the agent takes
async def chat(runner, session_id, text, user_id="student"):
    print(f"\n🧑 You: {text}")
    msg = types.Content(role="user", parts=[types.Part(text=text)])
    async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=msg):
        for part in (event.content.parts if event.content else []):
            if part.function_call:
                print(f"   🛠️  Agent decided to call → {part.function_call.name}({dict(part.function_call.args)})")
            if part.function_response:
                print(f"   📦 Tool returned        → {part.function_response.response}")
            if part.text and event.is_final_response():
                print(f"🤖 {event.author}: {part.text}")

session_service = InMemorySessionService()
runner = Runner(agent=hello_agent, app_name="hello_app", session_service=session_service)
session = await session_service.create_session(app_name="hello_app", user_id="student")
await chat(runner, session.id, "Hi! I just joined first year. I'm nervous 😅")


🧑 You: Hi! I just joined first year. I'm nervous 😅
🤖 hello_agent: Welcome to college! You are going to do amazing things here.


✏️ **Your turn:** change the `instruction` so the agent replies like a **pirate 🏴‍☠️**, or only in **Telugu / Hindi / Odia**. Re-run the cell. You just did *prompt engineering*!

---
## Part 3 · Give your agent TOOLS 🛠️ (20 min)

In ADK, **a tool is just a normal Python function**. The magic:
- The **function name + docstring** tell the LLM *what the tool does and when to use it*.
- The **type hints** (`day: str`) tell the LLM *what arguments to pass*.

The LLM never runs your code itself — it *asks* ADK to run it, gets the result, then writes the answer. This loop is called **Reason → Act → Observe** (ReAct).

We'll save the agent as a real project folder, because the **Dev UI**, **evaluation** and **Cloud Run** all use this folder layout:

```
campus_buddy/
├── __init__.py      ← tells Python this is a package
├── agent.py         ← the agent (must define root_agent)
├── .env             ← your API key (never commit!)
└── requirements.txt ← packages for Cloud Run
```

In [ ]:
import os
os.makedirs("campus_buddy", exist_ok=True)   # works on Windows, macOS, Linux and Colab

In [33]:
%%writefile campus_buddy/__init__.py
from . import agent

Overwriting campus_buddy/__init__.py


In [34]:
%%writefile campus_buddy/agent.py
"""Campus Buddy - a friendly helper agent for college students (built with Google ADK)."""
import os
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools import ToolContext

# Groq-hosted LLM via LiteLLM. Needs GROQ_API_KEY in the environment (or in .env).
# max_tokens keeps each call under Groq's free-tier output-tokens-per-minute limit;
# num_retries waits and retries automatically on 429 "rate limit" errors.
MODEL = LiteLlm(
    model=os.getenv("ADK_MODEL", "groq/qwen/qwen3.8-27b"),
    max_tokens=300,
    num_retries=5,
)

# ---------- Fake college data (in real life this would be a database or API) ----------
TIMETABLE = {
    "monday": ["09:00 Python Programming", "11:00 Engineering Maths", "14:00 Digital Logic Lab"],
    "tuesday": ["09:00 Physics", "11:00 Python Programming", "14:00 Communication Skills"],
    "wednesday": ["09:00 Engineering Maths", "11:00 Data Structures", "14:00 Sports"],
    "thursday": ["09:00 Data Structures", "11:00 Physics Lab", "14:00 Library Hour"],
    "friday": ["09:00 AI Club Workshop", "11:00 Engineering Maths", "14:00 Mentor Meeting"],
}
CANTEEN_MENU = {
    "monday": "Idli, Sambar, Lemon Rice",
    "tuesday": "Poori, Aloo Curry, Curd Rice",
    "wednesday": "Dosa, Chutney, Veg Biryani",
    "thursday": "Upma, Rajma Chawal, Buttermilk",
    "friday": "Pongal, Chicken Biryani, Gulab Jamun",
}


# ---------- TOOLS: plain Python functions. The docstring tells the LLM when to use them ----------
def get_timetable(day: str) -> dict:
    """Returns the class timetable for a given weekday.

    Args:
        day: Full weekday name, for example "Monday".
    """
    classes = TIMETABLE.get(day.strip().lower())
    if not classes:
        return {"status": "error", "message": f"No classes found for '{day}'. Try Monday to Friday."}
    return {"status": "success", "day": day.title(), "classes": classes}


def get_canteen_menu(day: str) -> dict:
    """Returns the canteen menu for a given weekday.

    Args:
        day: Full weekday name, for example "Friday".
    """
    menu = CANTEEN_MENU.get(day.strip().lower())
    if not menu:
        return {"status": "error", "message": f"Canteen is closed on '{day}'."}
    return {"status": "success", "day": day.title(), "menu": menu}


def calculate_cgpa(grade_points: list[float], credits: list[int]) -> dict:
    """Calculates the CGPA (credit-weighted average of grade points).

    Args:
        grade_points: Grade point of each subject, in order (for example [9, 8, 10]).
        credits: Credits of each subject, in the same order (for example [4, 3, 3]).
    """
    if len(grade_points) != len(credits) or not credits:
        return {"status": "error", "message": "Give one credit value for every grade point."}
    total = sum(g * c for g, c in zip(grade_points, credits))
    cgpa = round(total / sum(credits), 2)
    return {"status": "success", "cgpa": cgpa}


def add_reminder(task: str, tool_context: ToolContext) -> dict:
    """Saves a reminder for the student so it can be recalled later in the conversation.

    Args:
        task: What the student wants to be reminded about.
    """
    reminders = tool_context.state.get("reminders", [])
    reminders.append(task)
    tool_context.state["reminders"] = reminders  # <- writes into SESSION STATE
    return {"status": "success", "saved": task, "total_reminders": len(reminders)}


def list_reminders(tool_context: ToolContext) -> dict:
    """Lists all reminders the student has saved in this session."""
    return {"status": "success", "reminders": tool_context.state.get("reminders", [])}


# ---------- THE AGENT = model + instructions + tools ----------
root_agent = Agent(
    name="campus_buddy",
    model=MODEL,
    description="A friendly campus assistant for first and second year college students.",
    instruction="""
You are Campus Buddy, a friendly assistant for first and second year college students.

Rules:
- For class schedules, ALWAYS call get_timetable. For food, ALWAYS call get_canteen_menu.
- For CGPA questions, ALWAYS call calculate_cgpa. Never do the maths yourself.
- When the student asks you to remember something, call add_reminder.
  When they ask what they need to remember, call list_reminders.
- Pass days as full weekday names with a capital letter, like "Monday".
- Keep answers short, warm and clear. Never invent classes or menu items.
""",
    tools=[get_timetable, get_canteen_menu, calculate_cgpa, add_reminder, list_reminders],
)

Overwriting campus_buddy/agent.py


In [35]:
%%writefile campus_buddy/requirements.txt
google-adk
litellm>=1.84

Overwriting campus_buddy/requirements.txt


In [ ]:
# Save the key in the agent's .env file (used by `adk web`, `adk eval` and deploy)
with open("campus_buddy/.env", "w") as f:
    f.write(f'GROQ_API_KEY={os.environ["GROQ_API_KEY"]}\n')
for name in sorted(os.listdir("campus_buddy")):
    print(name)

🧠 **Predict first:** for *"What's my CGPA if I got 9, 8, 10 with credits 4, 3, 3?"* — which tool will the agent call, and with what arguments?

In [37]:
import importlib, campus_buddy.agent as cb
importlib.reload(cb)

session_service = InMemorySessionService()
runner = Runner(agent=cb.root_agent, app_name="campus_buddy", session_service=session_service)
session = await session_service.create_session(app_name="campus_buddy", user_id="student")

await chat(runner, session.id, "What classes do I have on Monday?")
await chat(runner, session.id, "What's my CGPA if I got 9, 8, 10 with credits 4, 3, 3?")
await chat(runner, session.id, "Is there biryani in the canteen this week? Which day?")


🧑 You: What classes do I have on Monday?
   🛠️  Agent decided to call → get_timetable({'day': 'Monday'})
   📦 Tool returned        → {'status': 'success', 'day': 'Monday', 'classes': ['09:00 Python Programming', '11:00 Engineering Maths', '14:00 Digital Logic Lab']}
🤖 campus_buddy: You have three classes on Monday:

- **09:00 – Python Programming**
- **11:00 – Engineering Maths**
- **14:00 – Digital Logic Lab**

Hope that helps! Let me know if you need anything else.

🧑 You: What's my CGPA if I got 9, 8, 10 with credits 4, 3, 3?
   🛠️  Agent decided to call → calculate_cgpa({'credits': [4, 3, 3], 'grade_points': [9, 8, 10]})
   📦 Tool returned        → {'status': 'success', 'cgpa': 9.0}
🤖 campus_buddy: Your CGPA is **9.0**. Nice work! 🎉

Let me know if you have any other questions.

🧑 You: Is there biryani in the canteen this week? Which day?
   🛠️  Agent decided to call → get_canteen_menu({'day': 'Monday'})
   🛠️  Agent decided to call → get_canteen_menu({'day': 'Tuesday'})
   🛠️  Ag

👀 Look at the 🛠️ lines: **the agent decided by itself** which tool to call and what arguments to pass. On the last question it may call `get_canteen_menu` **several times** — it's planning!

✏️ **Your turn (pick one):**
1. Add a tool `get_exam_date(subject: str) -> dict` with a small dictionary of exam dates. Add it to `tools=[...]` and ask *"When is my Physics exam?"*
2. Ask something no tool can answer (*"Who won IPL 2020?"*). What does the agent do?
3. Delete the docstring of `get_timetable` and re-run. Does the agent still use it correctly? **Why do docstrings matter?**

---
## Part 4 · Sessions, State & Events — the agent's memory 📒 (15 min)

| Concept | Meaning | Analogy |
|---|---|---|
| **Session** | one conversation thread | one WhatsApp chat |
| **Event** | every single step in that chat (your message, tool call, tool result, reply) | each message bubble |
| **State** | a small key-value scratchpad the agent can read/write | sticky notes on the chat |

Our `add_reminder` tool writes into **state** using `tool_context.state`.

In [39]:
await chat(runner, session.id, "Remind me to submit my Digital Logic lab record")



🧑 You: Remind me to submit my Digital Logic lab record
   🛠️  Agent decided to call → add_reminder({'task': 'Submit my Digital Logic lab record'})
   📦 Tool returned        → {'status': 'success', 'saved': 'Submit my Digital Logic lab record', 'total_reminders': 1}
🤖 campus_buddy: Got it! I've set a reminder for you: **Submit your Digital Logic lab record**. 👍

Let me know when you'd like to check your reminders!


In [40]:
s = await session_service.get_session(app_name="campus_buddy", user_id="student", session_id=session.id)
print("📒 STATE :", s.state)
print("🧾 EVENTS:", len(s.events), "events in this session\n")
for e in s.events:
    kind = ("tool call" if e.get_function_calls() else
            "tool result" if e.get_function_responses() else "text")
    print(f"  [{e.author:>12}] {kind}")

📒 STATE : {'reminders': ['Submit my Digital Logic lab record']}
🧾 EVENTS: 17 events in this session

  [        user] text
  [campus_buddy] tool call
  [campus_buddy] tool result
  [campus_buddy] text
  [        user] text
  [campus_buddy] tool call
  [campus_buddy] tool result
  [campus_buddy] text
  [        user] text
  [campus_buddy] tool call
  [campus_buddy] tool result
  [campus_buddy] text
  [        user] text
  [        user] text
  [campus_buddy] tool call
  [campus_buddy] tool result
  [campus_buddy] text


🧠 **Predict first:** if we create a **new session** and ask *"What do I need to remember?"* — will it remember the lab record?

In [41]:
new_session = await session_service.create_session(app_name="campus_buddy", user_id="student")
await chat(runner, new_session.id, "What do I need to remember?")


🧑 You: What do I need to remember?
   🛠️  Agent decided to call → list_reminders({})
   📦 Tool returned        → {'status': 'success', 'reminders': []}
🤖 campus_buddy: You don't have any reminders saved yet! If there's something you'd like to keep in mind — like a due date, an event, or a to-do — just let me know and I'll save it for you. 😊


👆 Empty! **State belongs to a session.** New chat = fresh sticky notes. (ADK also has `user:` state and long-term *Memory* services for remembering across sessions — a great thing to explore later.)

---
## Part 5 · ADK Dev UI — see inside the agent's brain 🔍 (15 min)

`adk web` starts a browser UI for chatting with and **debugging** your agent. Run it from the folder that **contains** `campus_buddy/`.

> 💻 **Best experience:** run this on your laptop: `pip install google-adk litellm` → put `GROQ_API_KEY=...` in `campus_buddy/.env` → `adk web` → open http://localhost:8000. The cells below try to open it inside Colab.

In [42]:
# Start `adk web` in the background (works on Colab, Linux, macOS and Windows)
import subprocess, sys, time

try:
    adk_web_proc.terminate()   # stop a previous run of this cell, if any
except NameError:
    pass

log = open("adk_web.log", "w")
adk_web_proc = subprocess.Popen(
    [sys.executable, "-m", "google.adk.cli", "web", "--host", "0.0.0.0", "--port", "8000", "."],
    stdout=log, stderr=subprocess.STDOUT,
)
time.sleep(8)
print("".join(open("adk_web.log", encoding="utf-8", errors="replace").readlines()[-5:]))

+-----------------------------------------------------------------------------+
| ADK Web Server shutting down...                                             |
+-----------------------------------------------------------------------------+

INFO:     Application shutdown complete.



In [43]:
try:
    from google.colab.output import eval_js   # only exists inside Google Colab
    url = eval_js("google.colab.kernel.proxyPort(8000)")
except ImportError:
    url = "http://localhost:8000/"             # running locally (VS Code / Jupyter)
print("👉 Open the ADK Dev UI:", url + "dev-ui/")

👉 Open the ADK Dev UI: http://localhost:8000/dev-ui/


### 🗺️ Tour of the Dev UI (do this together)
1. **Select app** (top-left) → `campus_buddy`. Ask: *"What's for lunch on Friday and what classes do I have that day?"*
2. **Events tab** → click each event: user message → `functionCall` → `functionResponse` → final text. These are the same events you printed in Part 4.
3. **Trace tab** → each row is one step with timing. Click a row to see **Event / Request / Response / Graph**. *Request* shows the **exact prompt + tool list** sent to the LLM. *Graph* draws the agent → tool flow.
4. **State tab** → ask *"Remind me to buy a lab coat"* and watch `reminders` appear live.
5. **Sessions tab** → start a **New session**, switch back and forth. State is per session!
6. **Eval tab** → we'll use it in Part 6 to save a chat as a **golden test case**.

🎯 **Class challenge:** who can find the *system instruction* inside the Trace → Request panel first? 🙋

---
## Part 6 · Evaluation with a GOLDEN DATASET 🥇 (20 min)

LLMs are **non-deterministic** — the same question can give slightly different answers. So how do we know our agent is *good*, and still good after we change it?

👉 We write a **golden dataset**: a list of questions with the **expected tool calls** (the *trajectory*) and a **reference answer**. ADK replays them and scores the agent:

| Metric | Checks | Our threshold |
|---|---|---|
| `tool_trajectory_avg_score` | Did it call the **right tool with the right arguments**? | 1.0 (must be exact) |
| `response_match_score` | How similar is the answer to the reference (ROUGE word overlap)? | 0.5 |

This is like an **exam answer key** for your agent 📝.

In [44]:
%%writefile campus_buddy/campus_buddy_golden.evalset.json
{
  "eval_set_id": "campus_buddy_golden",
  "name": "Campus Buddy golden dataset",
  "description": "Golden questions with the expected tool call and a reference answer.",
  "eval_cases": [
    {
      "eval_id": "monday_timetable",
      "conversation": [
        {
          "invocation_id": "inv-monday_timetable",
          "user_content": {
            "role": "user",
            "parts": [
              {
                "text": "What classes do I have on Monday?"
              }
            ]
          },
          "final_response": {
            "role": "model",
            "parts": [
              {
                "text": "On Monday you have Python Programming at 09:00, Engineering Maths at 11:00 and Digital Logic Lab at 14:00."
              }
            ]
          },
          "intermediate_data": {
            "tool_uses": [
              {
                "name": "get_timetable",
                "args": {
                  "day": "Monday"
                }
              }
            ],
            "intermediate_responses": []
          }
        }
      ],
      "session_input": {
        "app_name": "campus_buddy",
        "user_id": "student",
        "state": {}
      }
    },
    {
      "eval_id": "friday_menu",
      "conversation": [
        {
          "invocation_id": "inv-friday_menu",
          "user_content": {
            "role": "user",
            "parts": [
              {
                "text": "What's in the canteen on Friday?"
              }
            ]
          },
          "final_response": {
            "role": "model",
            "parts": [
              {
                "text": "On Friday the canteen is serving Pongal, Chicken Biryani and Gulab Jamun."
              }
            ]
          },
          "intermediate_data": {
            "tool_uses": [
              {
                "name": "get_canteen_menu",
                "args": {
                  "day": "Friday"
                }
              }
            ],
            "intermediate_responses": []
          }
        }
      ],
      "session_input": {
        "app_name": "campus_buddy",
        "user_id": "student",
        "state": {}
      }
    },
    {
      "eval_id": "cgpa_calc",
      "conversation": [
        {
          "invocation_id": "inv-cgpa_calc",
          "user_content": {
            "role": "user",
            "parts": [
              {
                "text": "My grade points are 9, 8 and 10 with credits 4, 3 and 3. What is my CGPA?"
              }
            ]
          },
          "final_response": {
            "role": "model",
            "parts": [
              {
                "text": "Your CGPA is 9.0. Great work!"
              }
            ]
          },
          "intermediate_data": {
            "tool_uses": [
              {
                "name": "calculate_cgpa",
                "args": {
                  "grade_points": [
                    9,
                    8,
                    10
                  ],
                  "credits": [
                    4,
                    3,
                    3
                  ]
                }
              }
            ],
            "intermediate_responses": []
          }
        }
      ],
      "session_input": {
        "app_name": "campus_buddy",
        "user_id": "student",
        "state": {}
      }
    }
  ]
}

Writing campus_buddy/campus_buddy_golden.evalset.json


In [45]:
%%writefile campus_buddy/test_config.json
{
  "criteria": {
    "tool_trajectory_avg_score": 1.0,
    "response_match_score": 0.5
  }
}

Writing campus_buddy/test_config.json


🧠 **Predict first:** will all 3 golden cases pass? Which metric is most likely to fail?

In [47]:
# Run `adk eval` and show only the key lines (pure Python filter — works on Windows, macOS, Linux and Colab)
import re, subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "google.adk.cli", "eval", "campus_buddy",
     "campus_buddy/campus_buddy_golden.evalset.json",
     "--config_file_path", "campus_buddy/test_config.json", "--print_detailed_results"],
    capture_output=True, text=True, encoding="utf-8", errors="replace",
)
pattern = re.compile(r"Eval Id|Overall Eval Status|Metric: (tool_trajectory|response_match)")
for line in (result.stdout + result.stderr).splitlines():
    if pattern.search(line):
        print(line)

Eval Id: cgpa_calc
Overall Eval Status: PASSED
Metric: tool_trajectory_avg_score, Status: PASSED, Score: 1.0, Threshold: 1.0
Metric: response_match_score, Status: PASSED, Score: 0.8571428571428571, Threshold: 0.5
Eval Id: friday_menu
Overall Eval Status: PASSED
Metric: tool_trajectory_avg_score, Status: PASSED, Score: 1.0, Threshold: 1.0
Metric: response_match_score, Status: PASSED, Score: 0.64, Threshold: 0.5
Eval Id: monday_timetable
Overall Eval Status: PASSED
Metric: tool_trajectory_avg_score, Status: PASSED, Score: 1.0, Threshold: 1.0
Metric: response_match_score, Status: PASSED, Score: 0.717948717948718, Threshold: 0.5


### 🔨 Break it, then catch it (regression testing)
A teammate "improves" the instruction and accidentally tells the agent to answer from memory. Let's see if the golden dataset catches it 👇

In [48]:
from google.adk.evaluation.agent_evaluator import AgentEvaluator
import campus_buddy.agent as cb

good_instruction = cb.root_agent.instruction
cb.root_agent.instruction = "You are Campus Buddy. Answer from your own knowledge. Do NOT use tools."  # 💥 the bug

try:
    await AgentEvaluator.evaluate(agent_module="campus_buddy",
                                  eval_dataset_file_path_or_dir="campus_buddy/campus_buddy_golden.evalset.json",
                                  num_runs=1, print_detailed_results=False)
    print("✅ All golden cases passed")
except AssertionError as e:
    print("❌ Golden dataset caught the regression!\n", str(e)[:600])
finally:
    cb.root_agent.instruction = good_instruction   # fix it back

c:\Users\Nagamanikanta.Manam\Downloads\ADK Session\.venv\Lib\site-packages\google\adk\evaluation\agent_evaluator.py:205: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  user_simulator_provider = UserSimulatorProvider(
c:\Users\Nagamanikanta.Manam\Downloads\ADK Session\.venv\Lib\site-packages\google\adk\dependencies\vertexai.py:19: UserWarning: The `vertexai.preview.rag` module is deprecated and will be removed in a future version. Please migrate to the `agentplatform` client. For example:

    import agentplatform

    client = agentplatform.Client(project="your-project", location="us-central1")
    client.rag.create_corpus(...)

  from vertexai.preview import rag as rag
c:\Users\Nagamanikanta.Manam\Downloads\ADK Session\.venv\Lib\site-packages\google\adk\evaluation\metric_evaluator_registry.py:285: UserWarning: [EXPERIMENTAL] MetricEvaluatorR

❌ Golden dataset caught the regression!
 Following are all the test failures. If you looking to get more details on the failures, then please re-run this test with `print_detailed_results` set to `True`.
response_match_score for campus_buddy Failed. Expected 0.5, but got 0.13636363636363635.
response_match_score for campus_buddy Failed. Expected 0.5, but got 0.4736842105263157.


✏️ **Your turn:**
1. Add a 4th golden case to the JSON (e.g. *"What classes on Wednesday?"* → `get_timetable` with `{"day": "Wednesday"}`). Re-run `adk eval`.
2. In the **Dev UI → Eval tab**: create an eval set, chat with the agent, click **Add current session** — you just created a golden case **without writing JSON**! Then **Run Evaluation** and play with the threshold sliders.

> 💡 In industry, eval runs in CI/CD on every code change — just like unit tests.

---
## Part 7 · Go LIVE on Cloud Run 🚀 (15 min · instructor demo)

**Cloud Run** runs your code in a container on Google's servers and gives you a public **https URL**. It scales to zero when nobody uses it.

> ⚠️ Needs a Google Cloud project **with billing enabled** (new accounts get free credits). Students without one: watch the demo, then try it at home. Easiest place to run these commands: **Google Cloud Shell** (https://shell.cloud.google.com) — `gcloud` is pre-installed.

**One-time setup**
```bash
pip install google-adk litellm
gcloud auth login
gcloud config set project YOUR_PROJECT_ID
gcloud services enable run.googleapis.com cloudbuild.googleapis.com artifactregistry.googleapis.com
```

**Deploy (one command!)** — run from the folder that contains `campus_buddy/`
```bash
adk deploy cloud_run \
  --project=YOUR_PROJECT_ID \
  --region=asia-south1 \
  --service_name=campus-buddy \
  --with_ui \
  ./campus_buddy
```
- `--with_ui` also deploys the Dev UI so you can chat in the browser. Drop it for an API-only service.
- When asked **"Allow unauthenticated invocations?"** → `y` for a public demo.
- Behind the scenes ADK writes a **Dockerfile**, **Cloud Build** builds the container, and **Cloud Run** runs it.

🔐 **Production tip:** don't ship the key in `.env`. Store it in Secret Manager and pass it with `-- --set-secrets=GROQ_API_KEY=GROQ_API_KEY:latest`.

In [ ]:
# OPTIONAL — needs a billing-enabled GCP project. Works in Colab AND locally (Windows / macOS / Linux)
import os, shutil, subprocess, sys

PROJECT_ID = "finance-app-508600"   # 👈 fill in
REGION = "asia-south1"              # Mumbai

def run(cmd):
    """Run a command and stream its output into the notebook (no shell tricks, so it works on Windows too)."""
    print("▶", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                            text=True, encoding="utf-8", errors="replace",
                            env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed (exit code {proc.returncode})")

if not PROJECT_ID:
    print("ℹ️ Skipped — set PROJECT_ID to deploy.")
else:
    gcloud = shutil.which("gcloud")   # finds gcloud.cmd on Windows
    if not gcloud:
        raise RuntimeError("gcloud not found — install the Google Cloud SDK: https://cloud.google.com/sdk/docs/install")

    try:
        from google.colab import auth    # Colab: log in with a popup
        auth.authenticate_user()
    except ImportError:                  # Local: use your existing gcloud login
        account = subprocess.run([gcloud, "auth", "list", "--filter=status:ACTIVE", "--format=value(account)"],
                                 capture_output=True, text=True).stdout.strip()
        if not account:
            raise RuntimeError("Not logged in. In a terminal run:  gcloud auth login  — then re-run this cell.")
        print("✅ Logged in as", account)

    run([gcloud, "config", "set", "project", PROJECT_ID])
    run([gcloud, "services", "enable", "run.googleapis.com", "cloudbuild.googleapis.com", "artifactregistry.googleapis.com"])
    # `-- --allow-unauthenticated` answers the "Allow unauthenticated invocations?" prompt (replaces `yes |`)
    run([sys.executable, "-m", "google.adk.cli", "deploy", "cloud_run",
         f"--project={PROJECT_ID}", f"--region={REGION}", "--service_name=campus-buddy", "--with_ui",
         "./campus_buddy", "--", "--allow-unauthenticated"])

**Test the live agent with its REST API** (works from any language or app!)

In [ ]:
APP_URL = ""   # 👈 paste the Service URL printed by the deploy, e.g. https://campus-buddy-xxxx.asia-south1.run.app
if APP_URL:
    !curl -s {APP_URL}/list-apps
    !curl -s -X POST {APP_URL}/apps/campus_buddy/users/u1/sessions/s1 -H "Content-Type: application/json" -d '{}'
    !curl -s -X POST {APP_URL}/run -H "Content-Type: application/json" -d '{"app_name":"campus_buddy","user_id":"u1","session_id":"s1","new_message":{"role":"user","parts":[{"text":"What classes on Friday?"}]}}'

🧹 **Clean up** so you're not billed:
```bash
gcloud run services delete campus-buddy --region=asia-south1
```

---
## Part 8 · Challenges 🏆

| Level | Challenge |
|---|---|
| 🥉 | Add `get_exam_date(subject)` and a golden case for it |
| 🥈 | Add a `bus_timings(stop: str)` tool and make the agent answer *"Which bus gets me to college before my first class on Monday?"* (needs **two** tools!) |
| 🥇 | Build a **multi-agent** team: a `study_coach` agent and a `canteen_agent`, with Campus Buddy delegating via `sub_agents=[...]` |
| 💎 | Deploy your agent to Cloud Run and share the URL with your friends |

### 📚 Keep learning
- ADK docs: https://adk.dev
- ADK samples: https://github.com/google/adk-samples
- Evaluation guide: https://adk.dev/evaluate/
- Cloud Run deploy: https://adk.dev/deploy/cloud-run/

### 🧠 Recap — the 5 words to remember
**Agent** = LLM + Tools + Memory + Goal · **Tool** = Python function + docstring · **Session/State/Event** = chat / sticky notes / message bubbles · **Golden dataset** = answer key · **Cloud Run** = your agent, live on the internet.